# Feature Engineering Pipeline

### Импорты и настройка окружения

In [1]:
import pandas as pd
import numpy as np
import gc
import os
import json
import mlflow
import logging
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import KFold

# Настройка логирования
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("Pipeline")

# Адрес базы данных
db_path = "sqlite:///../mlflow.db" 
mlflow.set_tracking_uri(db_path)

# Имя эксперимента
mlflow.set_experiment("Home_Credit_Default_Risk")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: sqlite:///../mlflow.db


### Загрузчик данных (DataLoader)

In [2]:
class DataLoader:
    def __init__(self, raw_dir: str = '../data/raw/', cache_dir: str = '../data/cache/'):
        self.raw_dir = raw_dir
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

    def load_table(self, table_name: str, use_cache: bool = True):
        cache_path = os.path.join(self.cache_dir, f"{table_name}.parquet")
        raw_path = os.path.join(self.raw_dir, f"{table_name}.csv")

        if use_cache and os.path.exists(cache_path):
            logger.info("Loading from cache | table=%s", table_name)
            return pd.read_parquet(cache_path)
        
        logger.info("Loading from CSV | table=%s", table_name)
        df = pd.read_csv(raw_path, engine="pyarrow")
        df.to_parquet(cache_path, index=False)
        return df

    def load_all(self, use_cache: bool = True):
        tables =['application_train', 'application_test', 'bureau', 'bureau_balance', 
                  'previous_application', 'installments_payments', 'POS_CASH_balance', 'credit_card_balance']
        return {name: self.load_table(name, use_cache=use_cache) for name in tables}

### Общие утилиты и препроцессинг

In [3]:
def one_hot_encoder(train: pd.DataFrame, test: pd.DataFrame) -> tuple:
    train = train.copy()
    test = test.copy()
    categorical_columns =[col for col in train.columns if train[col].dtype == 'object']
    if not categorical_columns:
        return train, test,[]

    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    enc.fit(train[categorical_columns])
    
    train_encoded = pd.DataFrame(enc.transform(train[categorical_columns]), columns=enc.get_feature_names_out(categorical_columns), index=train.index)
    test_encoded = pd.DataFrame(enc.transform(test[categorical_columns]), columns=enc.get_feature_names_out(categorical_columns), index=test.index)
    
    train = pd.concat([train.drop(columns=categorical_columns), train_encoded], axis=1)
    test = pd.concat([test.drop(columns=categorical_columns), test_encoded], axis=1)
    
    return train, test, list(enc.get_feature_names_out(categorical_columns))

def reduce_cardinality(train: pd.DataFrame, test: pd.DataFrame, cardinality_threshold: int = 10) -> tuple:
    train = train.copy()
    test = test.copy()
    modified_cols = []
    categorical_columns =[col for col in train.columns if train[col].dtype == 'object']
    
    for col in categorical_columns:
        if train[col].nunique() > cardinality_threshold:
            top_categories = train[col].value_counts().head(cardinality_threshold).index.tolist()
            train[col] = train[col].apply(lambda x: x if x in top_categories else 'Other')
            test[col] = test[col].apply(lambda x: x if x in top_categories else 'Other')
            modified_cols.append(col)
            
    return train, test, modified_cols

class ApplicationPreprocessor:
    def __init__(self):
        self.label_encoders = {}

    def fix_anomalies(self, df: pd.DataFrame):
        df = df.copy()
        if 'DAYS_EMPLOYED' in df.columns:
            df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
            df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
        if 'DAYS_LAST_PHONE_CHANGE' in df.columns:
            df['DAYS_LAST_PHONE_CHANGE'] = df['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan)
        if 'DAYS_BIRTH' in df.columns:
            df['DAYS_BIRTH'] = df['DAYS_BIRTH'].abs()
        df.replace({'XNA': np.nan, 'Unknown': np.nan}, inplace=True)
        return df

    def fit_transform_categorical(self, train: pd.DataFrame, test: pd.DataFrame):
        train = train.copy()
        test = test.copy()
        cat_cols = [col for col in train.columns if train[col].dtype == 'object']
        
        for col in cat_cols:
            le = LabelEncoder()
            le.fit(train[col])
            train[col] = le.transform(train[col])
            test[col] = test[col].apply(lambda x: x if x in le.classes_ else "UNKNOWN")
            le.classes_ = np.append(le.classes_, "UNKNOWN")
            test[col] = le.transform(test[col])
            self.label_encoders[col] = le
            
        return train, test

class PreviousPreprocessor:
    def fix_anomalies(self, df: pd.DataFrame):
        df = df.copy()
        for col in['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']:
            if col in df.columns:
                df[col] = df[col].replace(365243, np.nan)
        return df

class POSCashPreprocessor:
    def fix_anomalies(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.copy()

class CreditCardPreprocessor:
    def fix_anomalies(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if 'SK_ID_PREV' in df.columns:
            df.drop(columns=['SK_ID_PREV'], inplace=True)
        return df

class InstallmentsPreprocessor:
    def fix_anomalies(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.copy()

class PassThroughPreprocessor:
    def fix_anomalies(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.copy()

### Feature Engineering — Application

In [4]:
def build_application_features(df: pd.DataFrame):
    df = df.copy()
    for col in['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
        df[f'{col}_WAS_MISSING'] = df[col].isnull().astype(np.int8)

    df['CREDIT_ANNUITY_RATIO'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'] + 1e-5)
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
    
    df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['DAYS_EMPLOYED_PERCENT'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
    df['CREDIT_DOWNPAYMENT'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    
    ext_cols =['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCES_MEAN'] = df[ext_cols].mean(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_SOURCES_STD'] = df[ext_cols].std(axis=1)
    
    df['EMPLOYED_TO_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['CAR_TO_BIRTH_RATIO'] = df['OWN_CAR_AGE'] / df['DAYS_BIRTH']
    df['PHONE_TO_BIRTH_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH']
    
    return df

### Feature Engineering — Bureau

In [5]:
def aggregate_bureau_balance(bureau_balance_df: pd.DataFrame, bb_cat: list) -> pd.DataFrame:
    bureau_balance_df = bureau_balance_df.copy()
    bb_aggregations = {'MONTHS_BALANCE': ['min', 'max', 'size']}
    for col in bb_cat:
        bb_aggregations[col] = ['mean']

    bb_agg = bureau_balance_df.groupby('SK_ID_BUREAU').agg(bb_aggregations)
    bb_agg.columns = pd.Index([f"{col}_{stat.upper()}" for col, stat in bb_agg.columns.tolist()])
    return bb_agg

def create_bureau_row_features(bureau_df: pd.DataFrame, bb_agg: pd.DataFrame) -> pd.DataFrame:
    bureau_df = bureau_df.copy()
    bureau_df = bureau_df.join(bb_agg, how='left', on='SK_ID_BUREAU')
    bureau_df.drop('SK_ID_BUREAU', axis=1, inplace=True)
    bureau_df['BUREAU_DEBT_CREDIT_RATIO'] = (bureau_df['AMT_CREDIT_SUM_DEBT'] / (bureau_df['AMT_CREDIT_SUM'] + 1e-5))
    return bureau_df

def aggregate_bureau_features(bureau_df: pd.DataFrame, bureau_cat: list) -> pd.DataFrame:
    num_aggregations = {
        'DAYS_CREDIT': ['mean', 'max', 'min', 'var'],
        'DAYS_CREDIT_ENDDATE': ['mean', 'max'],
        'AMT_CREDIT_MAX_OVERDUE':['mean', 'max'],
        'AMT_CREDIT_SUM': ['mean', 'max', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['mean', 'max', 'sum'],
        'BUREAU_DEBT_CREDIT_RATIO': ['mean', 'max']
    }
    cat_aggregations = {col: ['mean'] for col in bureau_cat}

    bureau_agg = bureau_df.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    bureau_agg.columns = pd.Index([f"BURO_{col}_{stat.upper()}" for col, stat in bureau_agg.columns.tolist()])

    active = bureau_df[bureau_df['CREDIT_ACTIVE_Active'] == 1] if 'CREDIT_ACTIVE_Active' in bureau_df.columns else pd.DataFrame()
    if not active.empty:
        active_agg = active.groupby('SK_ID_CURR').agg({'DAYS_CREDIT': ['max']})
        active_agg.columns = pd.Index(['BURO_LAST_ACTIVE_DAYS_CREDIT_MAX'])
        bureau_agg = bureau_agg.join(active_agg, how='left')

    del bureau_df
    gc.collect()
    return bureau_agg

def build_bureau_features(bureau_df: pd.DataFrame, bureau_balance_df: pd.DataFrame, bb_cat: list, bureau_cat: list) -> pd.DataFrame:
    bb_agg = aggregate_bureau_balance(bureau_balance_df, bb_cat)
    bureau_rows = create_bureau_row_features(bureau_df, bb_agg)
    return aggregate_bureau_features(bureau_rows, bureau_cat)

### Feature Engineering — Previous, POS, Installments, Credit Card

In [6]:
# Previous
def create_previous_row_features(prev_df: pd.DataFrame) -> pd.DataFrame:
    prev_df = prev_df.copy()
    prev_df['ASKED_AMT_ID_CURR_DIFF'] = (prev_df['AMT_APPLICATION'] - prev_df['AMT_CREDIT'])
    prev_df['APPLICATION_CREDIT_RATIO'] = (prev_df['AMT_APPLICATION'] / (prev_df['AMT_CREDIT'] + 1e-5))
    prev_df['DOWNPAYMENT_CREDIT_RATIO'] = (prev_df['AMT_DOWN_PAYMENT'] / (prev_df['AMT_CREDIT'] + 1e-5))
    prev_df['ANNUITY_CREDIT_RATIO'] = (prev_df['AMT_ANNUITY'] / (prev_df['AMT_CREDIT'] + 1e-5))
    return prev_df

def aggregate_previous_features(prev_df: pd.DataFrame, prev_cat: list) -> pd.DataFrame:
    num_aggregations = {
        'AMT_ANNUITY': ['mean', 'max'], 'AMT_APPLICATION': ['mean', 'max'],
        'AMT_CREDIT': ['mean', 'max'], 'AMT_DOWN_PAYMENT': ['mean', 'max'],
        'AMT_GOODS_PRICE': ['mean', 'max'], 'HOUR_APPR_PROCESS_START': ['mean', 'max'],
        'RATE_DOWN_PAYMENT': ['mean', 'max'], 'DAYS_DECISION': ['mean', 'max'],
        'CNT_PAYMENT':['mean', 'sum'], 'ASKED_AMT_ID_CURR_DIFF': ['mean', 'max', 'sum'],
        'APPLICATION_CREDIT_RATIO': ['mean', 'max'], 'DOWNPAYMENT_CREDIT_RATIO': ['mean', 'max'],
        'ANNUITY_CREDIT_RATIO':['mean', 'max'],
    }
    cat_aggregations = {col: ['mean'] for col in prev_cat}

    prev_agg = prev_df.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    prev_agg.columns = pd.Index([f"PREV_{col}_{stat.upper()}" for col, stat in prev_agg.columns.tolist()])

    last_app = (prev_df.sort_values('DAYS_DECISION', ascending=False).groupby('SK_ID_CURR').first())
    last_cols =[c for c in last_app.columns if 'NAME_CONTRACT_STATUS' in c or 'PRODUCT_COMBINATION' in c]
    last_app = last_app[last_cols]
    last_app.columns =['PREV_LAST_' + c for c in last_app.columns]
    
    return prev_agg.join(last_app, how='left')

def build_previous_features(prev_df, prev_cat):
    return aggregate_previous_features(create_previous_row_features(prev_df), prev_cat)

# Pos cash
def build_pos_cash_features(pos_df: pd.DataFrame, pos_cat: list) -> pd.DataFrame:
    aggregations = {'MONTHS_BALANCE': ['max', 'mean', 'size'], 'SK_DPD': ['max', 'mean'], 'SK_DPD_DEF': ['max', 'mean']}
    for col in pos_cat: aggregations[col] = ['mean']
        
    pos_agg = pos_df.groupby('SK_ID_CURR').agg(aggregations)
    pos_agg.columns = pd.Index([f"POS_{col}_{stat.upper()}" for col, stat in pos_agg.columns.tolist()])
    
    active_col = 'NAME_CONTRACT_STATUS_Active'
    pos_agg['POS_ACTIVE_COUNT'] = pos_df[pos_df[active_col] == 1].groupby('SK_ID_CURR').size() if active_col in pos_df.columns else 0
    return pos_agg

# Installments
def build_installments_features(ins_df: pd.DataFrame) -> pd.DataFrame:
    ins_df = ins_df.copy()
    ins_df['PAYMENT_PERC'] = ins_df['AMT_PAYMENT'] / (ins_df['AMT_INSTALMENT'] + 1e-5)
    ins_df['PAYMENT_DIFF'] = ins_df['AMT_INSTALMENT'] - ins_df['AMT_PAYMENT']
    ins_df['DPD'] = (ins_df['DAYS_ENTRY_PAYMENT'] - ins_df['DAYS_INSTALMENT']).clip(lower=0)
    ins_df['DBD'] = (ins_df['DAYS_INSTALMENT'] - ins_df['DAYS_ENTRY_PAYMENT']).clip(lower=0)
    
    aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'], 'DPD':['max', 'mean', 'sum'], 'DBD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['max', 'mean', 'var'], 'PAYMENT_DIFF':['max', 'mean', 'sum', 'var'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'], 'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT': ['max', 'mean', 'sum']
    }
    
    ins_agg = ins_df.groupby('SK_ID_CURR').agg(aggregations)
    ins_agg.columns = pd.Index([f"INS_{col}_{stat.upper()}" for col, stat in ins_agg.columns.tolist()])
    ins_agg['INS_COUNT'] = ins_df.groupby('SK_ID_CURR').size()

    ins_1000_agg = ins_df[ins_df['DAYS_INSTALMENT'] > -1000].groupby('SK_ID_CURR').agg({'PAYMENT_DIFF': ['mean']})
    ins_1000_agg.columns = pd.Index(['INS_1000_PAYMENT_DIFF_MEAN'])
    ins_agg = ins_agg.join(ins_1000_agg, how='left')

    ins_365_agg = ins_df[ins_df['DAYS_INSTALMENT'] > -365].groupby('SK_ID_CURR').agg({'DPD': ['sum'], 'PAYMENT_DIFF': ['mean']})
    ins_365_agg.columns = pd.Index(['INS_365_DPD_SUM', 'INS_365_PAYMENT_DIFF_MEAN'])
    return ins_agg.join(ins_365_agg, how='left')

# Credit card
def build_credit_card_features(cc_df: pd.DataFrame, cc_cat: list) -> pd.DataFrame:
    exclude_cols =['SK_ID_CURR'] + cc_cat
    num_cols =[c for c in cc_df.columns if c not in exclude_cols]
    
    aggregations = {col:['min', 'max', 'mean', 'sum', 'var'] for col in num_cols}
    for col in cc_cat: aggregations[col] = ['mean']
        
    cc_agg = cc_df.groupby('SK_ID_CURR').agg(aggregations)
    cc_agg.columns = pd.Index([f"CC_{col}_{stat.upper()}" for col, stat in cc_agg.columns.tolist()])
    
    if 'CC_AMT_BALANCE_MEAN' in cc_agg.columns and 'CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN' in cc_agg.columns:
        cc_agg['CC_LIMIT_USE_MEAN'] = (cc_agg['CC_AMT_BALANCE_MEAN'] / (cc_agg['CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN'] + 1e-5))
    return cc_agg

### Оркестратор пайплайна

In [ ]:
class FeatureEngineeringPipeline:
    def __init__(self, raw_dir='data/raw', cache_dir='data/cache', use_cache=True, output_dir='data/processed'):
        self.loader = DataLoader(raw_dir=raw_dir, cache_dir=cache_dir)
        self.use_cache = use_cache
        self.output_dir = output_dir
        
        self.app_preprocessor = ApplicationPreprocessor()
        self.prev_preprocessor = PreviousPreprocessor()
        self.pos_preprocessor = POSCashPreprocessor()
        self.cc_preprocessor = CreditCardPreprocessor()
        self.inst_preprocessor = InstallmentsPreprocessor()
        self.generic_preprocessor = PassThroughPreprocessor()

    def _process_side_table(self, df, train_ids, preprocessor, table_name):
        logger.info(f"Preprocessing {table_name}...")
        df = preprocessor.fix_anomalies(df)
        
        if 'SK_ID_CURR' not in df.columns:
            raise KeyError(f"'SK_ID_CURR' is missing in {table_name}.")
            
        train_part = df[df['SK_ID_CURR'].isin(train_ids)]
        test_part = df[~df['SK_ID_CURR'].isin(train_ids)]
        
        train_part, test_part, _ = reduce_cardinality(train_part, test_part)
        train_part, test_part, cat_cols = one_hot_encoder(train_part, test_part)
        
        return pd.concat([train_part, test_part], ignore_index=True), cat_cols

    def run(self) -> pd.DataFrame:
        with mlflow.start_run(run_name="Feature_Engineering"):
            # Логируем входные параметры
            mlflow.log_params({
                "top_n_selection": 300,
                "cache_enabled": self.use_cache,
                "output_dir": self.output_dir
            })
            
            data = self.loader.load_all(use_cache=self.use_cache)
            
            # Application
            train_raw = self.app_preprocessor.fix_anomalies(data['application_train'])
            test_raw = self.app_preprocessor.fix_anomalies(data['application_test'])
            del data['application_train'], data['application_test']; gc.collect()
            
            train_ids = train_raw['SK_ID_CURR'].unique()
            train_app, test_app = self.app_preprocessor.fit_transform_categorical(train_raw, test_raw)
            del train_raw, test_raw; gc.collect()
            
            df = build_application_features(pd.concat([train_app, test_app], ignore_index=True))
            del train_app, test_app; gc.collect()
            
            # Bureau and Bureau Balance
            bureau_curr_map = data['bureau'][['SK_ID_BUREAU', 'SK_ID_CURR']].drop_duplicates()
            bureau, bureau_cat = self._process_side_table(data['bureau'], train_ids, self.generic_preprocessor, "Bureau")
            del data['bureau']
            
            bureau_balance = data['bureau_balance'].merge(bureau_curr_map, on='SK_ID_BUREAU', how='inner')
            del data['bureau_balance'], bureau_curr_map
            
            bb, bb_cat = self._process_side_table(bureau_balance, train_ids, self.generic_preprocessor, "Bureau Balance")
            del bureau_balance; gc.collect()

            bureau_features = build_bureau_features(bureau, bb, bb_cat, bureau_cat)
            del bureau, bb; gc.collect()

            # Previous apps
            prev, prev_cat = self._process_side_table(data['previous_application'], train_ids, self.prev_preprocessor, "Previous")
            del data['previous_application']
            previous_features = build_previous_features(prev, prev_cat)
            del prev; gc.collect()

            # Installments
            inst = self.inst_preprocessor.fix_anomalies(data['installments_payments'])
            del data['installments_payments']
            installments_features = build_installments_features(inst)
            del inst; gc.collect()

            # Pos cash
            pos, pos_cat = self._process_side_table(data['POS_CASH_balance'], train_ids, self.pos_preprocessor, "POS CASH")
            del data['POS_CASH_balance']
            pos_features = build_pos_cash_features(pos, pos_cat)
            del pos; gc.collect()

            # Credit card
            cc, cc_cat = self._process_side_table(data['credit_card_balance'], train_ids, self.cc_preprocessor, "Credit Card")
            del data['credit_card_balance']
            cc_features = build_credit_card_features(cc, cc_cat)
            del cc; gc.collect()
            
            # Merge всех фичей
            logger.info("Merging features...")
            df = df.merge(bureau_features, on='SK_ID_CURR', how='left')
            del bureau_features; gc.collect()
            
            df = df.merge(previous_features, on='SK_ID_CURR', how='left')
            del previous_features; gc.collect()
            
            df = df.merge(installments_features, on='SK_ID_CURR', how='left')
            del installments_features; gc.collect()
            
            df = df.merge(pos_features, on='SK_ID_CURR', how='left')
            del pos_features; gc.collect()
            
            df = df.merge(cc_features, on='SK_ID_CURR', how='left')
            del cc_features; gc.collect()
            
            # Feature selection
            df = self._select_features_ridge(df, top_n=300)
            self._save_results(df)
            
            # Логируем метрики и артефакты
            mlflow.log_metric("df_columns", df.shape[1])
            mlflow.log_metric("df_rows", df.shape[0])
            
            # Загружаем файлы в MLflow UI
            mlflow.log_artifact(os.path.join(self.output_dir, 'selected_features.json'))
            # Сохраняем head данных
            df.head(100).to_csv("sample_features.csv", index=False)
            mlflow.log_artifact("sample_features.csv")
            
            return df
        
    def _select_features_ridge(self, df: pd.DataFrame, top_n: int = 300) -> pd.DataFrame:
        logger.info(f"Feature selection via Ridge (top_n={top_n})")
        train_df = df[df['TARGET'].notnull()]
        X = train_df.drop(columns=['SK_ID_CURR', 'TARGET']).select_dtypes(include=['number'])
        y = train_df['TARGET']
        
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        X_prepared = pipeline.fit_transform(X)
        selector = SelectFromModel(Ridge(alpha=1.0), max_features=top_n, threshold=-np.inf)
        selector.fit(X_prepared, y)
        
        # Извлечение важности коэффициентов
        ridge_model = selector.estimator_ # Достаем сам обученный Ridge из селектора
        importance = np.abs(ridge_model.coef_)
        
        feature_importance = pd.DataFrame({'feature': X.columns, 'importance': importance}).sort_values(by='importance', ascending=False)
    
        # Сохранение и логирование важности признаков
        importance_path = os.path.join(self.output_dir, 'feature_importance.csv')
        feature_importance.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path)
        
        # Логируем ТОП-10 признаков
        top_10 = feature_importance.head(10).to_string()
        mlflow.log_text(top_10, "top_10_features.txt")
        
        selected_cols = X.columns[selector.get_support()].tolist()
        final_cols =['SK_ID_CURR', 'TARGET'] + selected_cols
        
        os.makedirs(self.output_dir, exist_ok=True)
        with open(os.path.join(self.output_dir, 'selected_features.json'), 'w') as f:
            json.dump(selected_cols, f, indent=2)
        mlflow.log_artifact(os.path.join(self.output_dir, 'selected_features.json'))
            
        return df[final_cols]

    def _save_results(self, df: pd.DataFrame) -> None:
        os.makedirs(self.output_dir, exist_ok=True)
        train = df[df['TARGET'].notnull()]
        test = df[df['TARGET'].isnull()].drop(columns=['TARGET'])
        
        train.to_parquet(os.path.join(self.output_dir, 'train_features.parquet'), index=False)
        test.to_parquet(os.path.join(self.output_dir, 'test_features.parquet'), index=False)
        logger.info(f"Saved: Train {train.shape}, Test {test.shape}")

### Запуск

In [8]:
pipeline = FeatureEngineeringPipeline(
    raw_dir='../data/raw/',
    cache_dir='../data/cache/',
    output_dir='../data/processed/',
    use_cache=True
)
final_dataset = pipeline.run()
display(final_dataset.head())

2026-05-04 17:12:00,890 | INFO | Loading from cache | table=application_train
2026-05-04 17:12:01,178 | INFO | Loading from cache | table=application_test
2026-05-04 17:12:01,212 | INFO | Loading from cache | table=bureau
2026-05-04 17:12:01,348 | INFO | Loading from cache | table=bureau_balance
2026-05-04 17:12:01,768 | INFO | Loading from cache | table=previous_application
2026-05-04 17:12:02,289 | INFO | Loading from cache | table=installments_payments
2026-05-04 17:12:02,875 | INFO | Loading from cache | table=POS_CASH_balance
2026-05-04 17:12:03,352 | INFO | Loading from cache | table=credit_card_balance
2026-05-04 17:12:07,478 | INFO | Preprocessing Bureau...
2026-05-04 17:12:11,160 | INFO | Preprocessing Bureau Balance...
2026-05-04 17:12:24,307 | INFO | Preprocessing Previous...
2026-05-04 17:12:50,530 | INFO | Preprocessing POS CASH...
2026-05-04 17:12:56,544 | INFO | Preprocessing Credit Card...
2026-05-04 17:13:00,643 | INFO | Merging features...
2026-05-04 17:13:04,707 | IN

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,CC_CNT_INSTALMENT_MATURE_CUM_MAX,CC_CNT_INSTALMENT_MATURE_CUM_MEAN,CC_CNT_INSTALMENT_MATURE_CUM_SUM,CC_CNT_INSTALMENT_MATURE_CUM_VAR,CC_SK_DPD_MAX,CC_SK_DPD_MEAN,CC_SK_DPD_SUM,CC_SK_DPD_DEF_MEAN,CC_SK_DPD_DEF_SUM,CC_SK_DPD_DEF_VAR
0,100002,1.0,0,1,0,0,202500.0,406597.5,24700.5,351000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0.0,0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0.0,1,1,1,0,67500.0,135000.0,6750.0,135000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0.0,0,0,0,0,135000.0,312682.5,29686.5,297000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100007,0.0,0,1,0,0,121500.0,513000.0,21865.5,513000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Запускаем в терминале `mlflow ui --backend-store-uri sqlite:///mlflow.db`